In [1]:
import boto3
import json
import re
from collections import defaultdict

# =====================
# CONFIG
# =====================

ENDPOINT_NAME = "indicbert-address-ner-v3"
runtime = boto3.client("sagemaker-runtime")

# =====================
# Endpoint Call
# =====================

def call_endpoint(text):
    payload = json.dumps({"inputs": text})
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=payload,
    )
    return json.loads(response["Body"].read())


# =====================
# Merge subwords
# =====================

def merge_subword_predictions(preds):

    results = []
    current = None
    buf = ""
    scores = []
    start = None
    end = None

    def flush():
        nonlocal current, buf, scores, start, end

        if current:
            results.append({
                "entity": current,
                "word": buf.strip(),
                "start": start,
                "end": end,
                "score": sum(scores) / len(scores),
            })

        current = None
        buf = ""
        scores = []
        start = None
        end = None

    for p in preds:

        tag = p["entity"]
        token = p["word"]
        clean = token.replace("▁", "")

        if tag.startswith("B-"):

            flush()
            current = tag[2:]
            buf = clean
            scores = [p["score"]]
            start = p["start"]
            end = p["end"]

        elif tag.startswith("I-") and current == tag[2:]:

            if token.startswith("▁"):
                buf += " "

            buf += clean
            end = p["end"]
            scores.append(p["score"])

        else:
            flush()

    flush()
    return results


# =====================
# Smart repair
# =====================

def smart_repair_entities(spans):

    fixed = []

    for s in spans:

        word = s["word"].strip()
        ent = s["entity"]

        new_ent = ent

        if re.fullmatch(r"\d{6}", word):
            new_ent = "PINCODE"

        elif re.search(r"[A-Za-z]", word) and not re.search(r"\d", word):
            if ent == "PINCODE":
                new_ent = "CITY"

        if re.search(r"(lp|flat|plot|house|unit|room|no\.?)", word, re.I):
            new_ent = "PREMISES_NO"

        if word.lower() in ["road", "lane", "street", "sarani", "avenue"]:
            new_ent = "ROAD"

        fixed.append({**s, "entity": new_ent})

    return fixed


# =====================
# Convert spans → structured address
# =====================

def spans_to_structured_address(spans):

    structured = defaultdict(list)

    for s in spans:

        entity = s["entity"]
        word = s["word"]

        if entity in ["PREMISES_NO", "ROAD", "CITY", "PINCODE"]:
            structured[entity].append(word)

    for k in structured:
        structured[k] = " ".join(structured[k])

    return dict(structured)


# =====================
# Final formatter
# =====================

def format_address_sentence(structured):

    def normalize(text):
        return text.strip().title()

    house = normalize(structured.get("PREMISES_NO", ""))
    road = normalize(structured.get("ROAD", ""))
    city = normalize(structured.get("CITY", ""))
    postcode = structured.get("PINCODE", "").strip()

    parts = []

    if house:
        parts.append(house)

    if road:
        parts.append(road)

    if city:
        parts.append(city)

    if postcode:
        parts.append(postcode)

    sentence = ", ".join(parts)
    sentence = re.sub(r"\s+", " ", sentence).strip()

    return sentence


# =====================
# Address Validation
# =====================

def validate_address(structured):

    house = structured.get("PREMISES_NO", "").strip()
    road = structured.get("ROAD", "").strip()
    city = structured.get("CITY", "").strip()
    postcode = structured.get("PINCODE", "").strip()

    if not (house or road or city or postcode):
        return False

    if postcode and not re.fullmatch(r"\d{6}", postcode):
        return False

    return True


# =====================
# MAIN PIPELINE
# =====================

def extract_and_format_address(text):

    preds = call_endpoint(text)

    merged = merge_subword_predictions(preds)

    repaired = smart_repair_entities(merged)

    structured = spans_to_structured_address(repaired)

    if not validate_address(structured):
        final_sentence = "invalid I/P"
    else:
        final_sentence = format_address_sentence(structured)

    if len(final_sentence) > 120:
        final_sentence = "invalid I/P"

    return structured, final_sentence


# =====================
# TEST (Real-time input)
# =====================

if __name__ == "__main__":

    user_text = input(
        "Please put address in the format { House No., Street, City & Pin Code }: "
    )

    structured, final_sentence = extract_and_format_address(user_text)

    print("\nFINAL ADDRESS:")
    print(final_sentence)

    print("\nSTRUCTURED:")
    print(structured)

Please put address in the format { House No., Street, City & Pin Code }:  26/1 MAHARANI INDIRA DEVI ROAD, KOLKATA 700034"



FINAL ADDRESS:
26/1, Kolkata, 700034

STRUCTURED:
{'PREMISES_NO': '26/1', 'CITY': 'kolkata', 'PINCODE': '700034'}
